# From Variant Effects to Pathway PGS in All of Us

**Made by Westerman Lab**

This tutorial shows how to convert variant-level genetic effects into a
participant-by-pathway polygenic score matrix in the All of Us controlled
workspace.

You will follow every scientific decision in the calculation:

$$
\text{variant effects}
\longrightarrow
\text{genes}
\longrightarrow
\text{pathways}
\longrightarrow
\text{participant-level pPGS}.
$$

Start with the built-in demonstration. It uses two example variants, two
example pathways, chromosome 21, and 100 All of Us participants. The demo is
only a technical check, not a scientific result. At the end, you will know
which inputs to replace for a complete analysis.

## Tutorial roadmap

Each step begins with the biological or statistical idea, followed by visible
code and a short QC output.

| Step | Question | Main output |
|---|---|---|
| 1. Choose evidence and participants | Which variant effects, p-value thresholds, and people will be analyzed? | Reproducible analysis settings |
| 2. Inspect weights and pathway files | Do the input columns, genes, and pathways mean what we expect? | Input QC and pathway preview |
| 3. Link variants to pathways | Which variants belong to each biological pathway? | Variant-to-pathway membership |
| 4. Match variants to All of Us | Are the same variants and alleles available in AoU genotypes? | Harmonized scoring input |
| 5. Clump, score, and interpret | Which variants remain after LD control, and what do the scores mean? | Participant pPGS, aggregate QC, figures, and report |

The scientific choices remain visible in the notebook. The reusable
implementations are stored in `pathway_prs_core.py`.

## The score calculated by this tutorial

For participant $i$, pathway $k$, and p-value threshold $\tau$,

$$
P_{ik}^{(\tau)}=
\sum_j
G_{ij}\widehat{\beta}_j
A_{jk}L_{jk}
\mathbf{1}(p_j\leq\tau).
$$

- $G_{ij}$ is the effect-allele dosage for variant $j$ in participant $i$.
- $\widehat{\beta}_j$ is the GWAS effect estimate. If the GWAS reports an odds
  ratio, its additive effect is represented on the log-odds scale.
- $A_{jk}=1$ when variant $j$ maps to a gene in pathway $k$.
- $L_{jk}=1$ when variant $j$ remains after pathway-specific LD clumping.
- $\mathbf{1}(p_j\leq\tau)$ retains variants that pass the selected GWAS
  p-value threshold.

Each GWAS variant has one p-value. The user chooses one or more thresholds.
The default is $\tau=1$, which keeps every mapped variant that remains after
LD clumping. Multiple thresholds produce a separate score at each threshold.

The custom SNP-to-gene option changes $A_{jk}$, not the trait effect
$\widehat{\beta}_j$. In this tutorial, the trait weight still comes from the
GWAS input.

## Setup: connect the notebook to the tutorial code

This cell only imports the analysis functions. It does not read participant
genotypes or calculate scores. Open `pathway_prs_core.py` to inspect the full
implementations used by every step.

In [ ]:
# Standard Python packages used for tables, figures, and file paths.
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Locate the tutorial folder whether the notebook was opened from the repo
# itself or from a common AoU home-directory location.
candidates = [
    Path.cwd(),
    Path.cwd() / "pathway-pgs-tutorial-AOU",
    Path.cwd() / "pathway-pgs-tutorial",
    Path.cwd() / "pathway_prs_tutorial",
    Path.home() / "pathway-pgs-tutorial-AOU",
    Path.home() / "pathway-pgs-tutorial",
    Path.home() / "pathway_prs_tutorial",
]
tutorial_dir = next(
    (path for path in candidates if (path / "pathway_prs_core.py").exists()),
    None,
)
if tutorial_dir is None:
    raise FileNotFoundError(
        "Open this notebook from the extracted pathway-PGS tutorial folder."
    )

sys.path.insert(0, str(tutorial_dir.resolve()))

# Import the exact reusable functions used in the teaching steps below.
from pathway_pgs_app import discover_local_inputs, launch_pathway_pgs_app
from pathway_prs_core import (
    WorkflowConfig,
    build_pathway_pgs_command,
    build_pathway_variant_union_bed,
    build_snp_set_from_variant_gene_mapping,
    build_variant_mapping_union_bed,
    command_as_shell,
    harmonize_gwas_ids_to_bed,
    infer_gwas_schema,
    plot_pathway_definition_qc,
    plot_pathway_pgs_results,
    read_gmt_summary,
    read_gtf_summary,
    run_pathway_pgs,
    summarize_aggregate_results,
    validate_inputs,
    write_markdown_report,
)


def show_qc(title, values, fields):
    """Display only the QC fields a beginner needs to interpret."""
    rows = []
    for key, label, meaning in fields:
        if key in values:
            rows.append(
                {"Check": label, "Result": values[key], "What it means": meaning}
            )
    print(title)
    display(pd.DataFrame(rows))


print("Setup complete")
print("Tutorial folder:", tutorial_dir.resolve())
print("Reusable implementation: pathway_prs_core.py")

## Step 1: Choose the variant evidence, participant cohort, and score settings

### Required scientific inputs

| Input | What it provides | Common alternatives |
|---|---|---|
| GWAS summary statistics | Variant ID, alleles, effect size, and p-value | FinnGen, GWAS Catalog studies, consortium GWAS, or another compatible study |
| Gene annotation or custom mapping | The variant-to-gene relationship | GRCh38/GRCh37 GTF; GTEx eQTL, regulatory, or chromatin-contact table |
| Pathway definition | The genes in each pathway | Reactome, GO Biological Process, WikiPathways, or custom GMT |
| Participant keep file | The AoU participants to score | A study cohort selected inside the controlled workspace |

### Parameters to understand

- `P_VALUE_THRESHOLDS`: `1.0` retains all mapped post-clumping variants;
  values such as `0.05` or `0.001` create more selective scores.
- `CLUMP_R2`: the maximum allowed LD with a retained index variant.
- `CLUMP_KB`: the genomic window used to search for correlated variants.
- `WINDOW_5_BP` and `WINDOW_3_BP`: strand-aware gene flanking windows used
  only for physical-distance mapping.

The demo executes automatically. When `RUN_MODE` is changed to `"full"`, data
preparation and scoring switch off until the user reviews the real files and
explicitly enables them.

In [ ]:
# ======================== USER-EDITABLE SETTINGS ========================

# Start with "demo". Change to "full" only after the demo succeeds.
RUN_MODE = "demo"                  # "demo" or "full"
GENOME_BUILD = "GRCh38"            # AoU v9 WGS coordinates

# Choose how variants are assigned to genes.
MAPPING_METHOD = "physical"         # "physical" or "custom"
PATHWAY_DATABASE = "Reactome"       # provenance label for the report

# Physical-mapping windows. These are ignored for custom mapping.
WINDOW_5_BP = 35_000                # upstream of each gene, strand-aware
WINDOW_3_BP = 10_000                # downstream of each gene, strand-aware

# Score construction. Add comma-separated values as a Python list if needed,
# for example [0.001, 0.05, 1.0]. The demo uses the PRSet-style default of 1.
P_VALUE_THRESHOLDS = [1.0]
CLUMP_KB = 1_000                    # LD search window in kilobases
CLUMP_R2 = 0.10                     # remove variants above this LD threshold
SCORE_METHOD = "sum"                # weighted dosage sum
THREADS = 4

# The demo runs end to end. Switching RUN_MODE to "full" turns both actions
# off automatically; set each to True only after reviewing the real inputs.
PREPARE_AOU_DATA = RUN_MODE == "demo"
RUN_SCORING = RUN_MODE == "demo"

# AoU person-level outputs must remain in the controlled workspace.
KEEP_PERSON_LEVEL_DATA_IN_AOU = True

# ========================================================================

home = Path.home()
demo_dir = tutorial_dir / "demo_data"
work_dir = home / "analysis" / "data" / "pathway_prs_adapter"
output_dir = home / "analysis" / "results" / "pathway_prs_tutorial"
discovered = discover_local_inputs()

if RUN_MODE == "demo":
    run_name = "pathway_pgs_quick_demo"
    gwas_file = demo_dir / "demo_gwas_chr21.tsv"
    gtf_file = demo_dir / "demo_chr21.gtf"
    gmt_file = demo_dir / "demo_pathways.gmt"
    custom_mapping_file = Path("")
    keep_file = work_dir / "quick_demo_100.keep"
    chromosomes = [21]
else:
    run_name = "pathway_pgs_full_analysis"
    gwas_file = Path(discovered["gwas"] or "/path/to/gwas.tsv.gz")
    gtf_file = Path(discovered["gtf"] or "/path/to/genes.gtf.gz")
    gmt_file = Path(discovered["gmt"] or "/path/to/pathways.gmt")
    custom_mapping_file = Path("/path/to/custom_snp_to_gene.tsv")
    keep_file = work_dir / "participants.keep"
    chromosomes = list(range(1, 23))

# Existing chromosome-wise AoU genotype files. The # is replaced by 1-22.
bed_pattern = (
    "/home/jupyter/workspace/vwb-aou-datasets-controlled-v9/v9/"
    "wgs/short_read/snpindel/acaf_threshold/plink_bed/"
    "acaf_threshold.chr#"
)

# Intermediate files created inside the controlled workspace.
union_bed = work_dir / "pathway_variant_union.bed1.tsv"
harmonized_gwas = work_dir / "gwas_for_aou_bed.tsv.gz"
target_list = work_dir / "aou_bed_target_prefixes.txt"
custom_snp_set = work_dir / "custom_pathway_snp_sets.gmt"

settings = pd.DataFrame(
    [
        ("Run mode", RUN_MODE, "Demo or complete analysis"),
        ("Genome build", GENOME_BUILD, "Must match GWAS, mapping, and AoU"),
        ("SNP-to-gene mapping", MAPPING_METHOD, "Defines pathway membership"),
        ("Pathway source", PATHWAY_DATABASE, "Biological gene sets"),
        ("P-value threshold(s)", str(P_VALUE_THRESHOLDS), "Variants retained"),
        ("LD window", f"{CLUMP_KB} kb", "Region searched for correlated variants"),
        ("LD threshold", CLUMP_R2, "Maximum retained pairwise r-squared"),
        ("Participants", "100 demo participants" if RUN_MODE == "demo" else keep_file, "Cohort to score"),
        ("Chromosomes", str(chromosomes), "Genotype partitions to read"),
    ],
    columns=["Setting", "Selected value", "Why it matters"],
)
display(settings)
print("Preparation enabled:", PREPARE_AOU_DATA)
print("Scoring enabled:", RUN_SCORING)

## Step 2: Inspect the variant weights and biological definitions

Before mapping or scoring, verify what every input column means.

### Variant weights

The program detects the GWAS columns for chromosome, position, variant ID,
effect allele, other allele, effect size, and p-value. The effect allele is
especially important because its dosage is multiplied by the effect size.

### Gene and pathway definitions

- A GTF file supplies gene coordinates for physical-distance mapping.
- A GMT file contains one pathway per row followed by its member genes.
- A custom mapping table can replace physical-distance mapping. It determines
  which gene receives each variant; it does not replace the GWAS effect size.

Replace the demo files with your own genome-build-matched inputs for a full
analysis. For example, AoU v9 WGS uses GRCh38 coordinates.

In [ ]:
# Detect the GWAS schema instead of assuming fixed column names.
schema = infer_gwas_schema(gwas_file)

role_meanings = {
    "chr": "Chromosome",
    "bp": "Genomic position",
    "snp": "Variant identifier",
    "a1": "Effect allele whose dosage is scored",
    "a2": "Other allele",
    "stat": "Variant effect weight (BETA or OR)",
    "p": "GWAS association p-value",
}
detected_columns = pd.DataFrame(
    [
        {
            "Required information": role_meanings.get(role, role),
            "Detected column": column,
        }
        for role, column in schema["gwas_columns"].items()
    ]
)
print("GWAS input check:", "PASS" if not schema["missing_required_fields"] else "REVIEW")
display(detected_columns)
print("Effect statistic:", schema["statistic_type"])
print("Unresolved required fields:", schema["missing_required_fields"] or "none")

# Read the pathway definitions without touching participant genotypes.
pathway_summary, pathway_gene_preview = read_gmt_summary(gmt_file)
print("\nPathway definition preview")
display(pathway_summary.head(10))
display(pathway_gene_preview.head(10))

if MAPPING_METHOD == "physical":
    gtf_qc = read_gtf_summary(gtf_file)
    show_qc(
        "Gene annotation check",
        gtf_qc,
        [(key, key.replace("_", " "), "GTF content check") for key in gtf_qc],
    )

# Visualize pathway sizes before scoring; very small or very large sets may
# require special interpretation.
figure = plot_pathway_definition_qc(pathway_summary)
display(figure)
plt.close(figure)

## Step 3: Link variants to genes and pathways

The membership indicator $A_{jk}$ is constructed in two stages:

$$
\text{variant }j \longrightarrow \text{gene }g
\longrightarrow \text{pathway }k.
$$

With physical-distance mapping, a variant maps to a gene when it falls inside
the gene body or the selected strand-aware flanking window. If $M_{jg}$ is
variant-to-gene membership and $B_{gk}$ is gene-to-pathway membership, then

$$
A_{jk}=\mathbf{1}\!\left(\sum_g M_{jg}B_{gk}>0\right).
$$

Choose `MAPPING_METHOD = "custom"` to provide another relationship, such as
an eQTL, enhancer-promoter, chromatin-contact, or experimentally curated
variant-to-gene table. Genes and pathways may overlap, so one variant may
contribute to multiple pathways. It is counted only once within a given
pathway.

In [ ]:
def make_config(base_path):
    """Collect every reviewed choice into one reproducible configuration."""
    current_schema = infer_gwas_schema(base_path)
    return WorkflowConfig(
        project_name=run_name,
        genome_build=GENOME_BUILD,
        base_gwas=str(base_path),
        base_separator=current_schema["separator"],
        gwas_columns=current_schema["gwas_columns"],
        statistic_type=current_schema["statistic_type"],
        target_type="bed",
        target_list=str(target_list),
        target_keep_file=str(keep_file),
        pathway_input_mode=(
            "gtf_gmt" if MAPPING_METHOD == "physical" else "snp_set"
        ),
        gtf_file=str(gtf_file),
        gmt_file=str(gmt_file),
        snp_set_file=str(custom_snp_set),
        window_5_bp=WINDOW_5_BP,
        window_3_bp=WINDOW_3_BP,
        pvalue_thresholds=P_VALUE_THRESHOLDS,
        clump_kb=CLUMP_KB,
        clump_r2=CLUMP_R2,
        score_method=SCORE_METHOD,
        threads=THREADS,
        prsice_r=discovered["wrapper"],
        prsice_binary=discovered["executable"],
        rscript=discovered["rscript"],
        output_dir=str(output_dir),
        output_prefix=run_name,
        controlled_workspace_acknowledged=KEEP_PERSON_LEVEL_DATA_IN_AOU,
    )


source_config = make_config(gwas_file)

mapping_plan = pd.DataFrame(
    [
        ("Variant to gene", MAPPING_METHOD, "GTF windows" if MAPPING_METHOD == "physical" else custom_mapping_file),
        ("Gene to pathway", PATHWAY_DATABASE, gmt_file),
        ("Membership output", "Deduplicated variant union", union_bed),
    ],
    columns=["Operation", "Selected method", "Input or output"],
)
display(mapping_plan)
print("No participant genotype dosage has been read in this step.")

## Step 4: Match pathway variants to All of Us genotypes

This step prepares the target data without copying the complete WGS dataset.

1. Build the deduplicated union of all variants needed by all pathways.
2. Match that union to AoU variant metadata using chromosome, position, and
   the allele pair.
3. Rewrite variant IDs into the form used by the AoU genotype files while
   retaining the GWAS effect allele and effect size.
4. Register the selected participant keep file and chromosome-wise AoU files.

The preparation QC answers two questions:

- **Mapping PASS:** were pathway genes and candidate variants identified?
- **Harmonization PASS:** were those variants found with compatible alleles
  in AoU?

Only metadata are read here. Participant dosages are first read during score
calculation. In demo mode, the first 100 IDs in the chromosome 21 target file
are used solely to verify the workflow.

In [ ]:
def ensure_demo_keep_file():
    """Create a 100-person keep file for technical demonstration only."""
    if RUN_MODE != "demo" or keep_file.exists():
        return
    prefix = bed_pattern.replace("#", "21")
    fam = Path(prefix + ".fam")
    if not fam.exists():
        raise FileNotFoundError(f"AoU chromosome 21 FAM not found: {fam}")

    # The first 100 IDs provide a reproducible technical test cohort. They are
    # not a random or scientifically selected analysis sample.
    ids = pd.read_csv(fam, sep=r"\s+", header=None, usecols=[0, 1], nrows=100)
    keep_file.parent.mkdir(parents=True, exist_ok=True)
    ids.to_csv(keep_file, sep="\t", header=False, index=False)


if not PREPARE_AOU_DATA:
    print("Preparation is OFF.")
    print("Review Steps 1-4, then set PREPARE_AOU_DATA = True in Step 1.")
else:
    ensure_demo_keep_file()
    work_dir.mkdir(parents=True, exist_ok=True)

    # Build A_jk: the pathway membership of every candidate variant.
    if MAPPING_METHOD == "physical":
        mapping_qc = build_pathway_variant_union_bed(source_config, union_bed)
    else:
        mapping_qc = build_variant_mapping_union_bed(
            source_config, custom_mapping_file, union_bed
        )

    show_qc(
        "Variant-to-pathway mapping QC",
        mapping_qc,
        [
            ("status", "Overall status", "PASS is required"),
            ("pathway_member_tokens", "Pathway gene entries", "Genes listed across pathways"),
            ("matched_gtf_genes", "Genes found in annotation", "Pathway genes with genomic coordinates"),
            ("unique_gwas_positions_in_pathway_windows", "Candidate variant positions", "Deduplicated mapped GWAS positions"),
            ("person_level_data_read", "Participant data read", "Must be False during mapping"),
        ],
    )

    # Match chromosome, position, and allele pair to AoU BIM metadata. Only
    # variant metadata are read; participant dosages are not read here.
    harmonization_qc = harmonize_gwas_ids_to_bed(
        source_config,
        bed_pattern=bed_pattern,
        chromosomes=chromosomes,
        candidate_bed1_file=union_bed,
        output_gwas=harmonized_gwas,
    )
    show_qc(
        "GWAS-to-AoU harmonization QC",
        harmonization_qc,
        [
            ("status", "Overall status", "PASS is required"),
            ("gwas_rows_inspected", "GWAS rows checked", "Input variants examined"),
            ("harmonized_gwas_rows", "Variants matched to AoU", "Rows available for scoring"),
            ("non_biallelic_or_non_snp_bim_rows", "Unsupported target records", "Expected to be zero or reviewed"),
            ("effect_alleles_or_effect_sizes_changed", "Effect data changed", "Expected to be False"),
            ("person_level_data_read", "Participant data read", "Must be False during harmonization"),
        ],
    )

    # Custom mappings are converted into pathway-specific SNP sets after the
    # variant IDs have been matched to AoU.
    if MAPPING_METHOD == "custom":
        snp_set_qc = build_snp_set_from_variant_gene_mapping(
            mapping_file=custom_mapping_file,
            gmt_file=gmt_file,
            output_file=custom_snp_set,
            harmonized_gwas_file=harmonized_gwas,
            harmonized_snp_column="SNP",
        )
        show_qc(
            "Custom pathway SNP-set QC",
            snp_set_qc,
            [(key, key.replace("_", " "), "Custom mapping output") for key in snp_set_qc],
        )

    # Register the existing chromosome files. No full WGS copy is created.
    prefixes = [bed_pattern.replace("#", str(chrom)) for chrom in chromosomes]
    target_list.write_text("\n".join(prefixes) + "\n")

    print("\nData preparation complete")
    print("Harmonized GWAS:", harmonized_gwas)
    print("AoU chromosome list:", target_list)

## Step 5: Control LD and calculate pathway scores

### 5A. Review pathway-specific LD clumping

Nearby variants can carry redundant information because of linkage
disequilibrium (LD). Within each pathway, the scoring engine uses GWAS
p-values to identify an index variant and removes nearby variants whose LD
with that index exceeds `CLUMP_R2` within `CLUMP_KB`.

This produces the indicator $L_{jk}$. A variant removed from one pathway may
still remain in another pathway because clumping is pathway-specific.

After clumping, the selected p-value threshold is applied. The notebook below
validates every input and prints the exact scoring command before execution.
For the demo, $\tau=1$, so p-values rank variants during clumping but do not
exclude additional variants by threshold.

In [ ]:
if not harmonized_gwas.exists():
    print("Prepared GWAS not found. Complete Step 4 first.")
    scoring_config = None
else:
    # Rebuild the configuration from the harmonized GWAS table.
    scoring_config = make_config(harmonized_gwas)
    checks, details = validate_inputs(scoring_config)

    print("Scoring readiness checks")
    display(checks)

    # This is the exact reproducible command that will be executed next.
    scoring_command = build_pathway_pgs_command(scoring_config)
    print("\nExact scoring command (review before running):\n")
    print(command_as_shell(scoring_command))

    # A dry run saves the configuration but does not calculate scores.
    dry_run_manifest = run_pathway_pgs(scoring_config, execute=False)
    print("\nDry-run status:", dry_run_manifest["status"])
    print("Saved configuration:", dry_run_manifest["config_path"])

### 5B. Calculate participant-level pPGS

The scoring engine now reads only the selected AoU participants and the
required pathway variants. For each participant and pathway, it computes the
weighted dosage sum

$$
P_{ik}^{(\tau)}=
\sum_{j\in S_k^{(\tau)}}G_{ij}\widehat{\beta}_j.
$$

The participant-level score file remains inside the controlled workspace.
Only aggregate QC is displayed by this notebook.

In [ ]:
if not RUN_SCORING:
    print("Scoring is OFF.")
    print("Review the readiness table and command, then set RUN_SCORING = True in Step 1.")
elif scoring_config is None:
    raise RuntimeError("Complete AoU variant preparation before scoring.")
elif not KEEP_PERSON_LEVEL_DATA_IN_AOU:
    raise PermissionError(
        "Confirm that all participant-level outputs remain inside the AoU workspace."
    )
else:
    # This is the only call that launches pathway-specific clumping and reads
    # selected participant dosages to calculate pPGS.
    run_manifest = run_pathway_pgs(scoring_config, execute=True)

    visible_manifest = {
        key: value
        for key, value in run_manifest.items()
        if key not in {"stdout", "stderr"}
    }
    print("Pathway scoring finished")
    display(pd.Series(visible_manifest, name="Result").to_frame())

## Interpret the score matrix and quality-control results

Each output column is one pathway score at one p-value threshold. Each row in
the protected score file is one participant.

The aggregate QC reports:

- `n`: participants with a score;
- `score_mean` and `score_sd`: score center and variability;
- `score_min` and `score_max`: observed score range;
- `all_finite`: whether the score contains only finite values.

A successful technical demo should produce finite scores for 100 participants
and variation in at least one example pathway. A pPGS is a relative genetic
score, not an absolute disease probability. Standardize it before comparing
effect sizes across pathways or cohorts, and adjust downstream association
models for appropriate covariates, including ancestry PCs.

In [ ]:
if scoring_config is None:
    print("No prepared scoring configuration is available.")
else:
    summary = summarize_aggregate_results(scoring_config)
    if summary.empty:
        print("No completed score output found. Run Step 5B first.")
    else:
        # Display only the columns needed for first-pass interpretation.
        preferred = [
            "score_name", "pathway", "threshold", "n", "n_snps",
            "score_mean", "score_sd", "score_min", "score_max", "all_finite",
            "prs_r2", "p_value",
        ]
        visible_columns = [column for column in preferred if column in summary.columns]

        print("Aggregate pathway-score QC")
        display(summary[visible_columns].head(30))

        finite_pass = (
            bool(summary["all_finite"].all())
            if "all_finite" in summary.columns
            else True
        )
        variable_count = (
            int((pd.to_numeric(summary["score_sd"], errors="coerce") > 0).sum())
            if "score_sd" in summary.columns
            else "not reported"
        )
        qc_overview = pd.DataFrame(
            [
                ("Pathway score columns", len(summary), "Number of pathway-threshold scores"),
                ("All values finite", finite_pass, "Must be True"),
                ("Nonconstant score columns", variable_count, "Should be greater than zero"),
            ],
            columns=["QC check", "Result", "Interpretation"],
        )
        display(qc_overview)

        # Save publication-ready raster and vector figures.
        figure = plot_pathway_pgs_results(summary)
        figure_dir = Path(scoring_config.output_dir) / "figures"
        figure_dir.mkdir(parents=True, exist_ok=True)
        png_path = figure_dir / "pathway_pgs_results.png"
        pdf_path = figure_dir / "pathway_pgs_results.pdf"
        figure.savefig(png_path, dpi=300, bbox_inches="tight")
        figure.savefig(pdf_path, bbox_inches="tight")
        display(figure)
        plt.close(figure)

        report = write_markdown_report(scoring_config, summary)
        print("Results complete")
        print("PNG figure:", png_path)
        print("PDF figure:", pdf_path)
        print("Reproducible report:", report)

## Move from the demo to a complete analysis

After the demo succeeds, make one change at a time and rerun from Step 1.

| What to replace | Demo value | Full-analysis action |
|---|---|---|
| Run mode | `"demo"` | Set `RUN_MODE = "full"` |
| GWAS | Two example variants | Supply a harmonized trait GWAS with alleles, effect size, and p-value |
| Variant-to-gene mapping | Physical distance | Keep a genome-matched GTF or provide a custom mapping table |
| Pathways | Two example pathways | Use Reactome, GO BP, WikiPathways, or a custom GMT |
| Participants | First 100 chromosome 21 IDs | Create an AoU keep file for the intended cohort |
| Chromosomes | Chromosome 21 | Use chromosomes 1-22 |
| Thresholds | `[1.0]` | Prespecify the primary threshold and optional sensitivity thresholds |
| Execution | Automatic for demo | Set the two execution switches to `True` only after reviewing Steps 1-5 |

For multiple thresholds, for example
`P_VALUE_THRESHOLDS = [0.001, 0.05, 1.0]`, the workflow creates a separate
pathway score at each threshold. Do not select a best threshold using the same
phenotype sample in which performance will be reported; use prespecification,
cross-validation, or an independent validation cohort.

## Congratulations! You have successfully completed the pathway PGS tutorial. 🎉

## The guided interface

The guided interface provides a convenient way to repeat the same pPGS workflow after reviewing the code above. It uses the same functions and parameters introduced in this tutorial.

In [ ]:
# The guided interface.
app = launch_pathway_pgs_app()
display(app)